Car Price Prediction - Model Training Script
Dataset: CarDekho (8128 records)

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

1. Load Data

In [ ]:
print("📦 Loading dataset...")
df = pd.read_csv("D:\\car price predection\\data\\car_data.csv")
print(f"   Shape: {df.shape}")

: 

2. Feature Engineering

In [3]:
print("\n🔧 Feature Engineering...")

# Car age from year
df["car_age"] = 2024 - df["year"]

# Extract brand from name
df["brand"] = df["name"].apply(lambda x: x.split()[0])

# Drop original name and year columns
df.drop(columns=["name", "year"], inplace=True)

# Convert numeric columns and fill missing values with median
for col in ["mileage(km/ltr/kg)", "engine", "max_power", "seats"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col].fillna(df[col].median(), inplace=True)

print(f"   Features after engineering: {df.columns.tolist()}")


🔧 Feature Engineering...
   Features after engineering: ['selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage(km/ltr/kg)', 'engine', 'max_power', 'seats', 'car_age', 'brand']


3. Encode Categoricals

In [4]:
print("\n🏷️  Encoding categorical features...")

cat_cols = ["fuel", "seller_type", "transmission", "owner", "brand"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"   {col}: {len(le.classes_)} classes")


🏷️  Encoding categorical features...
   fuel: 4 classes
   seller_type: 3 classes
   transmission: 2 classes
   owner: 5 classes
   brand: 32 classes


 4. Train / Test Split

In [5]:
df.dropna(inplace=True)
print(f"   Rows after dropna: {len(df)}")

X = df.drop(columns=["selling_price"])
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\n📊 Train: {len(X_train)} | Test: {len(X_test)}")

   Rows after dropna: 8128

📊 Train: 6502 | Test: 1626


5. Train Models

In [6]:
print("\n🚀 Training models...")

models = {
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=5, random_state=42
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=200, max_depth=None, random_state=42, n_jobs=-1
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2, "model": model}
    print(f"   {name}: MAE=₹{mae:,.0f} | RMSE=₹{rmse:,.0f} | R²={r2:.4f}")


🚀 Training models...
   HistGradientBoosting: MAE=₹75,740 | RMSE=₹147,426 | R²=0.9668
   RandomForest: MAE=₹67,963 | RMSE=₹140,128 | R²=0.9700


6. Select Best Model

In [7]:
best_name = max(results, key=lambda k: results[k]["R2"])
best_model = results[best_name]["model"]
print(f"\n🏆 Best model: {best_name} (R²={results[best_name]['R2']:.4f})")

# Feature importances (RandomForest has it, HistGB may not)
if hasattr(best_model, "feature_importances_"):
    feat_imp = dict(zip(X.columns, best_model.feature_importances_))
else:
    feat_imp = {col: 1/len(X.columns) for col in X.columns}
feat_imp_sorted = dict(sorted(feat_imp.items(), key=lambda x: x[1], reverse=True))


🏆 Best model: RandomForest (R²=0.9700)


7. Save Artifacts

In [8]:
print("\n💾 Saving model artifacts...")
joblib.dump(best_model, "C:\\Users\\THARANITHARAN S\\Downloads\\files (2)\\backend\\model.pkl")
joblib.dump(encoders, "C:\\Users\\THARANITHARAN S\\Downloads\\files (2)\\backend\\encoders.pkl")
joblib.dump(list(X.columns), "C:\\Users\\THARANITHARAN S\\Downloads\\files (2)\\backend\\feature_names.pkl")

# Save metadata
meta = {
    "best_model": best_name,
    "metrics": {
        k: {"MAE": v["MAE"], "RMSE": v["RMSE"], "R2": v["R2"]}
        for k, v in results.items()
    },
    "feature_importance": feat_imp_sorted,
    "brands": sorted(encoders["brand"].classes_.tolist()),
    "fuel_types": encoders["fuel"].classes_.tolist(),
    "seller_types": encoders["seller_type"].classes_.tolist(),
    "transmission_types": encoders["transmission"].classes_.tolist(),
    "owner_types": encoders["owner"].classes_.tolist(),
    "feature_columns": list(X.columns),
    "price_stats": {
        "min": int(y.min()),
        "max": int(y.max()),
        "mean": int(y.mean()),
        "median": int(y.median()),
    },
}

with open("C:\\Users\\THARANITHARAN S\\Downloads\\files (2)\\backend\\model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Training complete! All artifacts saved.")
print(f"\n📈 Final Metrics ({best_name}):")
print(f"   MAE  : ₹{results[best_name]['MAE']:,.0f}")
print(f"   RMSE : ₹{results[best_name]['RMSE']:,.0f}")
print(f"   R²   : {results[best_name]['R2']:.4f}")



💾 Saving model artifacts...
✅ Training complete! All artifacts saved.

📈 Final Metrics (RandomForest):
   MAE  : ₹67,963
   RMSE : ₹140,128
   R²   : 0.9700
